<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/06_LLM_Generation_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch pandas numpy
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import pickle
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
# 1. Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Mounted at /content/drive


In [ ]:
print("📥 Đang load dữ liệu ánh xạ...")

# 1. Load thông tin thống kê (để biết ID nào là Course, ID nào là Concept)
stats = {}
with open(os.path.join(PROCESSED_PATH, 'entity_stats.txt'), 'r') as f:
    for line in f:
        k, v = line.strip().split('=')
        stats[k] = int(v)

user_num = stats['user_num']
course_num = stats['course_num']
# ID Ranges
course_start = user_num
course_end = user_num + course_num

📥 Đang load dữ liệu ánh xạ...


In [ ]:
# 2. Load Mapping (ID Số -> Mã Gốc)
with open(os.path.join(PROCESSED_PATH, 'course_map.pkl'), 'rb') as f:
    id_to_course_code = {v: k for k, v in pickle.load(f).items()}

try:
    with open(os.path.join(PROCESSED_PATH, 'course_map.pkl'), 'rb') as f:
        id_to_concept_code = {v: k for k, v in pickle.load(f).items()}
except:
    print("⚠️ Không tìm thấy concept_map.pkl, sẽ hiển thị ID số.")
    id_to_concept_code = {}

In [ ]:
# 3. Hàm dịch ID sang Tên (Để đưa vào Prompt)
def get_node_name(node_id):
    node_id = int(node_id)
    if node_id < course_start:
        return f"Người học (ID {node_id})"
    elif course_start <= node_id < course_end:
        local_id = node_id - course_start
        code = id_to_course_code.get(local_id, f"Course_{local_id}")
        return f"Khóa học [{code}]"
    else:
        local_id = node_id - course_end
        code = id_to_concept_code.get(local_id, f"Concept_{local_id}")
        # Làm đẹp tên Concept (bỏ tiền tố K_)
        clean_name = code.replace("K_", "").replace("_", " ")
        return f"Khái niệm '{clean_name}'"

print("✅ Đã nạp xong từ điển dữ liệu!")

✅ Đã nạp xong từ điển dữ liệu!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Tên mô hình
model_name = "Qwen/Qwen2.5-7B-Instruct"

print("⏳ Đang tải Qwen 2.5 (khoảng 1-2 phút)...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Or "fp4"
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config, # Pass the config here
    trust_remote_code=True
)
print("✅ Qwen 2.5 đã sẵn sàng phục vụ!")

⏳ Đang tải Qwen 2.5 (khoảng 1-2 phút)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen 2.5 đã sẵn sàng phục vụ!


In [ ]:
def explain_path_with_llm(path_list):
    """
    path_list: Danh sách ID các node trong đường dẫn giải thích
    Ví dụ: [96, 6508, 27675, 6690] (Lấy từ Output máy local)
    """
    # 1. Dịch ID sang Tên
    try:
        user_name = get_node_name(path_list[0])
        history_course = get_node_name(path_list[1])
        bridge_concept = get_node_name(path_list[2])
        target_course = get_node_name(path_list[3])
    except IndexError:
        return "Lỗi: Đường dẫn không đủ độ dài để giải thích."

    # 2. Tạo Prompt
    prompt = f"""
    Đóng vai một trợ lý học tập AI thông minh và tận tâm.
    Dựa trên dữ liệu phân tích hành vi học tập sau:
    - {user_name} đã hoàn thành: {history_course}
    - Khóa học này trang bị kiến thức về: {bridge_concept}
    - Hệ thống đề xuất bước tiếp theo là: {target_course}

    Nhiệm vụ: Hãy viết một đoạn văn ngắn (2-3 câu) bằng tiếng Việt để giải thích lý do đề xuất này cho người học.
    Văn phong: Khích lệ, chuyên nghiệp, nhấn mạnh vào việc mở rộng kiến thức liên quan đến {bridge_concept}.
    """

    # 3. Gửi cho Qwen
    messages = [
        {"role": "system", "content": "Bạn là chuyên gia tư vấn giáo dục."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=200,
        temperature=0.7
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # Lấy phần trả lời của Assistant (cắt bỏ prompt)
    if "assistant\n" in response:
        return response.split("assistant\n")[-1].strip()
    return response

print("✅ Hàm giải thích đã sẵn sàng!")

✅ Hàm giải thích đã sẵn sàng!


In [ ]:
# --- NHẬP DỮ LIỆU TỪ MÁY LOCAL VÀO ĐÂY ---
# Ví dụ từ kết quả của bạn: User 96 -> Course 6508 -> Concept 27675 -> Course 6690
#local_result_path = [96, 6508, 27675, 6690]
local_result_path = [95, 6505, 5600, 6550]
print("🤖 Đang viết lời giải thích...")
explanation = explain_path_with_llm(local_result_path)

print("\n" + "="*50)
print("💡 KẾT QUẢ HIỂN THỊ TRÊN GIAO DIỆN NGƯỜI DÙNG:")
print("="*50)
print(explanation)

🤖 Đang viết lời giải thích...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



💡 KẾT QUẢ HIỂN THỊ TRÊN GIAO DIỆN NGƯỜI DÙNG:
Chào bạn ID 95! Sau khi hoàn thành khóa học "Khái niệm về Người học" (ID 5600), hệ thống của chúng tôi đã nhận ra rằng bạn đã có nền tảng vững chắc về lĩnh vực này. Bây giờ, chúng tôi đề xuất bạn tham gia khóa học mới "[C_course-v1:TsinghuaX+00740123_X+sp]" vì nó sẽ giúp bạn mở rộng kiến thức về người học, đưa ra những hiểu biết sâu hơn và ứng dụng hiệu quả hơn những gì bạn đã học trước đây. Chúc bạn một trải nghiệm học tập thú vị!


In [ ]:
# --- LƯU KẾT QUẢ VÀO FILE ---
output_filename = os.path.join(PROCESSED_PATH, 'llm_explanation.txt')

with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(explanation)

print(f"\n✅ Đã lưu lời giải thích vào file: {output_filename}")


✅ Đã lưu lời giải thích vào file: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/llm_explanation.txt


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import math
import pickle
from collections import defaultdict
from tqdm import tqdm
from google.colab import drive

# 1. KẾT NỐI DRIVE & CẤU HÌNH
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data/processed/')

# 2. ĐỊNH NGHĨA LẠI CLASS (Bắt buộc để load model)
class MOOCEnvironment:
    def __init__(self, data_path):
        self.ent_emb = np.load(os.path.join(data_path, 'entity_embeddings.npy'))
        self.rel_emb = np.load(os.path.join(data_path, 'relation_embeddings.npy'))
        kg = pd.read_csv(os.path.join(data_path, 'kg_final.txt'), sep=' ', header=None)
        self.graph = defaultdict(list)
        for _, row in kg.iterrows():
            self.graph[int(row[0])].append((int(row[1]), int(row[2])))

        with open(os.path.join(data_path, 'entity_stats.txt'), 'r') as f:
            stats = {l.split('=')[0]: int(l.split('=')[1]) for l in f}
        self.course_start = stats['user_num']
        self.course_end = stats['user_num'] + stats['course_num']
        self.visited = set()

    def reset(self, uid):
        self.state = uid
        self.path = [(uid, -1)]
        self.visited = {uid}
        return self.get_state()

    def get_state(self):
        uid = self.path[0][0]
        return np.concatenate([self.ent_emb[uid], self.ent_emb[self.state]])

    def get_valid_actions(self):
        acts = self.graph[self.state]
        return [(r, n) for r, n in acts if n not in self.visited]

    def step(self, action):
        r, n = action
        self.state = n
        self.path.append((n, r))
        self.visited.add(n)
        return self.get_state(), 0, False, self.path

# --- SỬA LẠI CLASS POLICYNET (Dùng l1/l2 thay vì fc1/fc2) ---
class PolicyNet(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        # Sửa tên biến fc1 -> l1, fc2 -> l2 để khớp với file đã lưu
        self.l1 = nn.Linear(s_dim + a_dim, 128)
        self.l2 = nn.Linear(128, 1)

    def forward(self, state, action_emb):
        x = torch.cat([state.unsqueeze(1).expand(-1, action_emb.size(1), -1), action_emb], -1)
        # Sửa fc1 -> l1
        x = F.elu(self.l1(x))
        # Sửa fc2 -> l2
        return self.l2(x).squeeze(-1)

# 3. LOAD MODEL & DATA
print("⏳ Đang load Model & Data...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = MOOCEnvironment(PROCESSED_PATH)
net = PolicyNet(200, 100).to(device)

# Load weights an toàn
model_path = os.path.join(PROCESSED_PATH, 'policy_net.pth')
try:
    net.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
except:
    net.load_state_dict(torch.load(model_path, map_location=device))
net.eval()

# Load Train History (để lọc khóa cũ)
train_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'train.txt'), sep=' ', header=None, names=['u','c','l'])
train_df['c'] += env.course_start
user_hist = train_df.groupby('u')['c'].apply(set).to_dict()

# Load Test Targets
test_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'test.txt'), sep=' ', header=None, names=['u','c','l'])
test_df['c'] += env.course_start
test_target = test_df.groupby('u')['c'].apply(set).to_dict()

# 4. CHẠY ĐÁNH GIÁ (INFERENCE LOOP)
metrics = {'p':[], 'r':[], 'ndcg':[], 'hr':[]}
TOP_K = 10
MAX_STEPS = 4
case_studies = [] # Lưu lại đường dẫn để dùng cho File 06

print(f"🚀 Bắt đầu đánh giá trên {len(test_target)} users...")

# Để demo nhanh, ta chạy trên 100 user đầu tiên. Muốn chạy hết hãy bỏ [:100]
target_users = list(test_target.keys())[:100]

for uid in tqdm(target_users):
    recs = defaultdict(int)
    hist = user_hist.get(uid, set())
    paths = {} # Lưu đường dẫn đến từng course

    # Sampling 50 lần/user
    for _ in range(50):
        state = env.reset(uid)
        for _ in range(MAX_STEPS):
            valid = env.get_valid_actions()
            if not valid: break

            # Chọn hành động
            act_vecs = []
            for r, n in valid: act_vecs.append(env.rel_emb[r] + env.ent_emb[n])
            act_t = torch.tensor(np.array(act_vecs)).float().to(device).unsqueeze(0)
            state_t = torch.tensor(state).float().to(device).unsqueeze(0)

            with torch.no_grad():
                probs = F.softmax(net(state_t, act_t), 1)
                idx = torch.multinomial(probs, 1).item()

            state, _, _, path = env.step(valid[idx])
            node = path[-1][0]

            if env.course_start <= node < env.course_end:
                if node not in hist:
                    recs[node] += 1
                    paths[node] = path # Lưu lại đường dẫn
                    break

    # Tính toán chỉ số
    top_k = sorted(recs, key=recs.get, reverse=True)[:TOP_K]
    if not top_k:
        for k in metrics: metrics[k].append(0)
        continue

    truth = test_target[uid]
    hits = len(set(top_k) & truth)

    metrics['p'].append(hits/TOP_K)
    metrics['r'].append(hits/len(truth))
    metrics['hr'].append(1 if hits > 0 else 0)

    dcg = sum([1/math.log2(i+2) for i, x in enumerate(top_k) if x in truth])
    idcg = sum([1/math.log2(i+2) for i in range(min(len(truth), TOP_K))])
    metrics['ndcg'].append(dcg/idcg if idcg > 0 else 0)

    # Lưu Case Study (nếu tìm thấy đúng)
    for item in top_k:
        if item in truth:
            case_studies.append({
                'uid': uid,
                'target_course': item,
                'path': paths[item]
            })
            break

# 5. XUẤT KẾT QUẢ & LƯU FILE FINAL_RESULTS.TXT
final_hr = np.mean(metrics['hr'])
final_ndcg = np.mean(metrics['ndcg'])
final_p = np.mean(metrics['p'])
final_r = np.mean(metrics['r'])

print("\n" + "="*40)
print(f"✅ Hit Rate: {final_hr:.4f}")
print(f"✅ NDCG:     {final_ndcg:.4f}")
print(f"✅ Precision:{final_p:.4f}")
print(f"✅ Recall:   {final_r:.4f}")
print("="*40)

# A. Lưu Case Studies dạng .pkl (cho File 06 dùng)
pkl_path = os.path.join(PROCESSED_PATH, 'case_studies.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(case_studies, f)
print(f"💾 Đã lưu dữ liệu Case Studies (raw) vào: {pkl_path}")

# B. Lưu Báo cáo chi tiết dạng Text (final_results.txt) - CẬP NHẬT THEO YÊU CẦU
txt_path = os.path.join(PROCESSED_PATH, 'final_results.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("==================================================\n")
    f.write("       KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (EVALUATION REPORT)\n")
    f.write("==================================================\n\n")

    f.write("1. METRICS (TRÊN TẬP TEST):\n")
    f.write(f"- Hit Rate@10:  {final_hr:.4f}\n")
    f.write(f"- NDCG@10:      {final_ndcg:.4f}\n")
    f.write(f"- Precision@10: {final_p:.4f}\n")
    f.write(f"- Recall@10:    {final_r:.4f}\n\n")

    f.write(f"2. DANH SÁCH CASE STUDIES THÀNH CÔNG ({len(case_studies)} trường hợp):\n")
    f.write("(Format: User -> [Path IDs] -> Target Course)\n")
    f.write("-" * 50 + "\n")

    for idx, case in enumerate(case_studies):
        f.write(f"Case #{idx+1}:\n")
        f.write(f"  User ID: {case['uid']}\n")
        f.write(f"  Target Course ID: {case['target_course']}\n")
        f.write(f"  Reasoning Path (Node IDs): {case['path']}\n")
        f.write("-" * 30 + "\n")

print(f"📄 Đã xuất báo cáo chi tiết vào: {txt_path}")
print("👉 Bạn có thể tải file 'final_results.txt' về để làm báo cáo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Đang load Model & Data...
🚀 Bắt đầu đánh giá trên 6486 users...


100%|██████████| 100/100 [00:31<00:00,  3.17it/s]



✅ Hit Rate: 0.2600
✅ NDCG:     0.0765
✅ Precision:0.0320
✅ Recall:   0.1137
💾 Đã lưu dữ liệu Case Studies (raw) vào: /content/drive/MyDrive/01. THESIS/My suggestion/data/processed/case_studies.pkl
📄 Đã xuất báo cáo chi tiết vào: /content/drive/MyDrive/01. THESIS/My suggestion/data/processed/final_results.txt
👉 Bạn có thể tải file 'final_results.txt' về để làm báo cáo.
